In [ ]:
# CELL 0: Setup & Imports
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    RandomizedSearchCV, cross_validate, StratifiedKFold
)
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, roc_auc_score, matthews_corrcoef,
    average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve, auc,
    classification_report
)

print("Imports successful")



Imports successful


In [ ]:
# CELL 1: Load & Inspect Data
# Col 0 = SMILES, Col 1 = labels, Col 2+ =  chemical Descriptors or ECFP FPs

TRAIN_CSV = '/content/train.csv'      #provide the precalculated
TEST_CSV  = '/content/test.csv'

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

feature_cols = [col for col in train_df.columns if col not in ['smiles', 'labels']]

X_train = train_df[feature_cols]
y_train = train_df['labels']
X_test  = test_df[feature_cols]
y_test  = test_df['labels']

print(f"Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")
print(f"Fingerprint bits: {X_train.shape[1]}")
print(f"\nTrain class distribution:\n{y_train.value_counts()}")
print(f"\nTest class distribution:\n{y_test.value_counts()}")

In [ ]:
# CELL 2: Hyperparameter Tuning  (scored on MCC)
param_grid = dict(
    n_estimators=[100, 300, 500, 800],
    max_depth=[None, 10, 20],
    min_samples_split=[2, 5, 10],
    max_features=['sqrt', 'log2', 0.3]
)

search = RandomizedSearchCV(
    RandomForestClassifier(
        criterion='entropy',
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    param_grid,
    n_iter=30,
    scoring='matthews_corrcoef',
    cv=5,
    random_state=42,
    n_jobs=-1,
    refit=True
)
search.fit(X_train, y_train)

print(f"Best hyperparameters : {search.best_params_}")
print(f"Best CV MCC (search) : {search.best_score_:.3f}")

In [ ]:
# CELL 3: Train Final RF  +  5-Fold CV
cv_strategy     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring_metrics = ['accuracy', 'f1', 'roc_auc', 'precision', 'recall', 'matthews_corrcoef']

cv_results = cross_validate(
    RandomForestClassifier(
        criterion='entropy',
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
        **search.best_params_
    ),
    X_train, y_train,
    cv=cv_strategy,
    scoring=scoring_metrics
)

print("5-Fold CV on Training Set")
for metric in scoring_metrics:
    m = cv_results[f'test_{metric}'].mean()
    s = cv_results[f'test_{metric}'].std()
    print(f"  {metric.replace('_', ' ').title():>22s}: {m:.3f} ± {s:.3f}")

model    = search.best_estimator_
y_pred   = model.predict(X_test.values)
y_proba  = model.predict_proba(X_test.values)[:, 1]

print("\nTraining complete")

In [ ]:
# CELL 4: Model Evaluation and Essential Metrics
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

prec_curve, rec_curve, _ = precision_recall_curve(y_test, y_proba)
pr_auc = auc(rec_curve, prec_curve)

metrics = {
    "Accuracy":           accuracy_score(y_test, y_pred),
    "Balanced Accuracy":  balanced_accuracy_score(y_test, y_pred),
    "F1-Score (macro)":   f1_score(y_test, y_pred, average='macro'),
    "F1-Score (class 1)": f1_score(y_test, y_pred, pos_label=1),
    "Sensitivity (TPR)":  tp / (tp + fn) if (tp + fn) > 0 else 0,
    "Specificity (TNR)":  tn / (tn + fp) if (tn + fp) > 0 else 0,
    "MCC":                matthews_corrcoef(y_test, y_pred),
    "ROC-AUC":            roc_auc_score(y_test, y_proba),
    "PR-AUC":             pr_auc,
    "TP": tp, "TN": tn, "FP": fp, "FN": fn
}

print("=" * 45)
print("   CLASSIFICATION RESULTS of Random Forest")
print("=" * 45)
for k, v in metrics.items():
    val = f"{v:.4f}" if isinstance(v, float) else str(v)
    print(f"  {k:<28}: {val}")

print("\n" + classification_report(y_test, y_pred,
      target_names=["Non-inhibitor (0)", "inhibitor (1)"], digits=4))

# Save metrics
pd.DataFrame([metrics]).T.rename(columns={0: "Value"}).to_csv('rf_metrics.csv')
print("Saved: rf_metrics.csv")